In [2]:
import os
import stat

# ---------------------------------------------------------------------------
# Generates ALL batch folders needed to cover a full run (e.g. 5 ns), each
# batch being a fixed chunk (e.g. 500 ps), for BOTH systems (Interface and
# Water), for ALL models (DPMD/NequIP/BPNN) automatically in one run of this
# script -- no prompt for which model to use.
#
# Batch 1 starts from the system's starting configuration; every batch after
# that read_restarts from the previous batch's lammps.restart file. Each
# batch writes its own restart file at the end, so if power/the job dies
# mid-run, you only ever need to resubmit from the last COMPLETED batch,
# losing at most one chunk.
#
# "model" (BPNN/NequIP/DPMD) only selects which frozen-graph directory to
# pull from -- every model uses pair_style deepmd.
#
# Mass / type map differs by system, matching each system's actual data file:
#   Interface (graphene-water): 1=O(15.999) 2=H(1.00794) 3=C(12.0107)
#   Water (pure water):         1=H(2.014)  2=O(15.999)
#
# WATER batches use the "simple" style, which mirrors the original reference
# water script exactly (plugin load, neighbor 2.0/neigh_modify, single fix
# nvt tau=0.1, no equilibration staging, dcd dump, literal values).
# INTERFACE batches use the "complex" style (variables, momentum zeroing,
# staged equilibration runs, custom dump w/ thermo_output.txt logging).
# ---------------------------------------------------------------------------

ROOT = "."  # relative to wherever this script is run from (TransferLearning_Corrected_Myself/)
TEMP_BY_SYSTEM = {"interface": 330, "water": 300}  # water lowered to 300 per request
DT = 0.0005  # ps, fixed
SYSTEMS = ["Interface", "Water"]  # generated automatically, no per-system prompt
MODELS = ["DPMD"]  # generated automatically, no per-model prompt

command_width = 15

unum = input("Enter the username: ").strip()

try:
    total_time_ps = float(input("Enter TOTAL run length in ps (e.g. 5000 for 5 ns): "))
    batch_time_ps = float(input("Enter batch chunk length in ps (e.g. 500): "))
    timeout_str = input("Enter the timer timeout PER BATCH in H:MM:SS format (e.g., 23:55:00): ")
    graph_num = int(input("Enter the number of graphs/models considered: "))
    script = input("Do you want the submission script for CPU or GPU version: ").strip().upper()
except ValueError:
    print("Invalid input. Please enter valid numbers or use the correct format.")
    exit()

if total_time_ps % batch_time_ps != 0:
    print(f"Error: total_time_ps ({total_time_ps}) must be an exact multiple of batch_time_ps ({batch_time_ps}).")
    exit()

num_batches = int(total_time_ps // batch_time_ps)
batch_steps = round(batch_time_ps / DT)


def mass_lines_for(system_name):
    if system_name.lower() == "interface":
        return [(1, "15.999000"), (2, "1.00794"), (3, "12.0107")]
    else:  # water
        return [(1, "2.014"), (2, "15.999")]


def write_batch(batch_idx, run_steps, combo, combo_dir, graph_files, system_lmp_path, mass_lines, style, temp):
    batch_label = f"batch_{batch_idx}"
    is_start = (batch_idx == 1)
    output_dir = os.path.join(combo_dir, batch_label)
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, "in.lammps")
    system_name_lbl = f"{combo}_{batch_label}"

    graph_files_rel = [os.path.relpath(p, start=output_dir) for p in graph_files]

    if is_start:
        source_path_rel = os.path.relpath(system_lmp_path, start=output_dir) if system_lmp_path else ""
    else:
        prev_restart = os.path.join(combo_dir, f"batch_{batch_idx - 1}", "lammps.restart")
        source_path_rel = os.path.relpath(prev_restart, start=output_dir)

    if style == "simple":
        # Mirrors the original reference water script structure exactly: plugin
        # load, neighbor 2.0/neigh_modify, single fix nvt (tau=0.1), no
        # settle/equilibrate staging, dcd dump, literal (non-variable) values.
        with open(output_file_path, 'w') as f:
            f.write(f"{'plugin load':<{command_width}} libdeepmd_lmp.so\n\n")
            f.write(f"{'units':<{command_width}} metal\n\n")
            f.write(f"{'boundary':<{command_width}} p p p\n\n")
            f.write(f"{'atom_style':<{command_width}} atomic\n\n")
            f.write(f"{'neighbor':<{command_width}} 2.0 bin\n\n")
            f.write(f"{'neigh_modify':<{command_width}} every 10 delay 0 check no\n\n")

            if is_start:
                f.write(f"{'read_data':<{command_width}} {source_path_rel}\n\n")
            else:
                f.write(f"{'read_restart':<{command_width}} {source_path_rel}\n\n")

            for type_id, mval in mass_lines:
                f.write(f"{'mass':<{command_width}} {type_id} {mval}\n")
            f.write("\n")

            pair_style_components = ["pair_style", "deepmd", *graph_files_rel]
            if graph_num > 1:
                pair_style_components.extend(["out_file", "md.out", "out_freq", "100"])
            f.write(" ".join(pair_style_components) + "\n\n")

            f.write(f"{'pair_coeff':<{command_width}} * *\n\n")
            if is_start:
                f.write(f"{'velocity':<{command_width}} all create {temp} 12345678\n\n")
            f.write(f"{'fix':<{command_width}} 1 all nvt temp {temp} {temp} 0.1\n\n")
            f.write(f"{'timestep':<{command_width}} 0.0005\n\n")
            f.write(f"{'thermo_style':<{command_width}} custom step pe ke etotal temp press vol\n\n")
            f.write(f"{'thermo':<{command_width}} 100\n\n")
            f.write(f"{'dump':<{command_width}} 1 all dcd 100 {system_name_lbl}.dcd\n\n")
            f.write(f"{'timer timeout':<{command_width}} {timeout_str} every 100\n\n")
            f.write(f"{'run':<{command_width}} {run_steps}\n\n")
            f.write(f"{'write_restart':<{command_width}} lammps.restart\n")

        print(f"Successfully generated {output_file_path}")
        _write_submission_files(output_dir, system_name_lbl)
        return

    with open(output_file_path, 'w') as f:
        f.write(f"{'variable':<{command_width}} TEMP            equal {temp}\n")
        f.write(f"{'variable':<{command_width}} NSTEPS          equal {run_steps}\n")
        f.write(f"{'variable':<{command_width}} THERMO_FREQ     equal 100\n")
        f.write(f"{'variable':<{command_width}} DUMP_FREQ       equal 1\n")
        f.write(f"{'variable':<{command_width}} dt              equal {DT}\n")
        f.write(f"{'variable':<{command_width}} TAU_T           equal 100*${{dt}}\n")
        f.write(f"{'variable':<{command_width}} restart         equal {0 if is_start else 1}\n\n")

        f.write(f"{'units':<{command_width}} metal\n")
        f.write(f"{'boundary':<{command_width}} p p p\n")
        f.write(f"{'atom_style':<{command_width}} atomic\n\n")

        if is_start:
            f.write(f"{'read_data':<{command_width}} {source_path_rel}\n\n")
        else:
            f.write(f"{'read_restart':<{command_width}} {source_path_rel}\n\n")

        for type_id, mval in mass_lines:
            f.write(f"{'mass':<{command_width}} {type_id} {mval}\n")
        f.write("\n")

        pair_style_components = ["pair_style", "deepmd", *graph_files_rel]
        if graph_num > 1:
            pair_style_components.extend(["out_file", "md.out", "out_freq", "100"])
        f.write(" ".join(pair_style_components) + "\n")
        f.write(f"{'pair_coeff':<{command_width}} * *\n\n")

        f.write(f"{'neighbor':<{command_width}} 1.0 bin\n")
        f.write(f"{'timestep':<{command_width}} ${{dt}}\n")
        f.write(f"{'thermo':<{command_width}} ${{THERMO_FREQ}}\n")
        f.write(f"{'thermo_style':<{command_width}} custom step temp pe ke etotal pxx pyy pzz press vol density\n")
        f.write(f"{'thermo_modify':<{command_width}} line multi\n\n")

        if is_start:
            f.write(f"{'velocity':<{command_width}} all create ${{TEMP}} 73294324 dist gaussian\n")
            f.write(f"{'fix':<{command_width}} mzero all momentum 1000 linear 1 1 1\n")
            f.write(f"{'fix':<{command_width}} 1 all nvt temp ${{TEMP}} ${{TEMP}} ${{TAU_T}}\n")
            f.write(f"{'run':<{command_width}} 5000\n")
            f.write(f"{'unfix':<{command_width}} 1\n\n")

            for var, expr in [("s", "step"), ("t", "temp"), ("pe", "pe"), ("ke", "ke"), ("et", "etotal"),
                               ("pxx", "pxx"), ("pyy", "pyy"), ("pzz", "pzz"), ("press", "press"),
                               ("vol", "vol"), ("rho", "density")]:
                f.write(f"variable {var} equal {expr}\n")
            f.write(
                'fix myOutput all print ${THERMO_FREQ} '
                '"${s} ${t} ${pe} ${ke} ${et} ${pxx} ${pyy} ${pzz} ${press} ${vol} ${rho}" '
                'file thermo_output.txt screen no\n\n'
            )

            f.write(f"{'fix':<{command_width}} 2 all nvt temp ${{TEMP}} ${{TEMP}} ${{TAU_T}}\n")
            f.write(f"{'run':<{command_width}} 100000\n")
            f.write(f"{'unfix':<{command_width}} 2\n\n")
        else:
            for var, expr in [("s", "step"), ("t", "temp"), ("pe", "pe"), ("ke", "ke"), ("et", "etotal"),
                               ("pxx", "pxx"), ("pyy", "pyy"), ("pzz", "pzz"), ("press", "press"),
                               ("vol", "vol"), ("rho", "density")]:
                f.write(f"variable {var} equal {expr}\n")
            f.write(
                'fix myOutput all print ${THERMO_FREQ} '
                '"${s} ${t} ${pe} ${ke} ${et} ${pxx} ${pyy} ${pzz} ${press} ${vol} ${rho}" '
                'file thermo_output.txt screen no append yes\n\n'
            )

        f.write(f"{'fix':<{command_width}} 3 all nvt temp ${{TEMP}} ${{TEMP}} ${{TAU_T}}\n")
        f.write(f"{'dump':<{command_width}} 1 all custom ${{DUMP_FREQ}} water.lammpstrj id type x y z\n")
        if not is_start:
            f.write(f"{'dump_modify':<{command_width}} 1 sort id append yes\n")
        else:
            f.write(f"{'dump_modify':<{command_width}} 1 sort id\n")
        f.write(f"{'timer timeout':<{command_width}} {timeout_str} every 100\n")
        f.write(f"{'run':<{command_width}} ${{NSTEPS}}\n")
        f.write(f"{'write_restart':<{command_width}} lammps.restart\n")

    print(f"Successfully generated {output_file_path}")
    _write_submission_files(output_dir, system_name_lbl)


def _write_submission_files(output_dir, system_name_lbl):
    sub_file_path = os.path.join(output_dir, "lmp.sub")
    if script == "CPU":
        with open(sub_file_path, 'w') as f:
            f.write("#!/bin/bash\n")
            f.write(f"#SBATCH --job-name={system_name_lbl}\n")
            f.write("#SBATCH --nodes=2\n")
            f.write("#SBATCH --ntasks-per-node=48\n")
            f.write("#SBATCH --partition=standard\n")
            f.write(f"#SBATCH --time={timeout_str}\n")
            f.write("#SBATCH --mail-type=end\n")
            f.write(f"#SBATCH --mail-user={unum}@smail.iitm.ac.in\n")
            f.write("export OMP_NUM_THREADS=2\n")
            f.write("mpirun -np 96 $HOME/.conda/envs/deepmd/bin/lmp -in in.lammps\n")
    else:
        with open(sub_file_path, 'w') as f:
            f.write("#!/bin/bash\n")
            f.write(f"#SBATCH --job-name={system_name_lbl}\n")
            f.write("#SBATCH --nodes=1\n")
            f.write("#SBATCH --ntasks-per-node=1\n")
            f.write("#SBATCH --gres=gpu:1\n")
            f.write("#SBATCH --partition=gpu\n")
            f.write(f"#SBATCH --time={timeout_str}\n")
            f.write("#SBATCH --mail-type=end\n")
            f.write(f"#SBATCH --mail-user={unum}@smail.iitm.ac.in\n\n")
            f.write("srun $HOME/.conda/envs/deepmd_gpu/bin/lmp -in in.lammps\n")
    print(f"Successfully generated {sub_file_path}")

    sh_path = os.path.join(output_dir, "run_job.sh")
    with open(sh_path, 'w') as f:
        f.write("#!/bin/bash\n")
        f.write("cd \"$(dirname \"$0\")\"\n")
        f.write("sbatch lmp.sub\n")
    os.chmod(sh_path, os.stat(sh_path).st_mode | stat.S_IEXEC)


def generate_for_system(model, system_name):
    combo = f"{model}_{system_name}"
    graph_dirz = os.path.join(ROOT, "Frozen_Graphs", combo)
    graph_files = [os.path.join(graph_dirz, f"{i+1}/graph{i+1}.pb") for i in range(graph_num)]
    for path in graph_files:
        print(f"[{combo}] Generated graph file path: {path}")

    starting_config_dir = os.path.join(ROOT, "Long_Run", "Starting_Configurations")
    if system_name.lower() == "interface":
        system_lmp_path = os.path.join(starting_config_dir, "Graphene_Water_DPMD_Validation_file.lmp")
        style = "complex"
    else:
        system_lmp_path = os.path.join(starting_config_dir, "Water_DPMD_Validation.lmp")
        style = "simple"

    mass_lines = mass_lines_for(system_name)
    temp = TEMP_BY_SYSTEM[system_name.lower()]

    combo_dir = os.path.join(ROOT, "Long_Run", combo)
    os.makedirs(combo_dir, exist_ok=True)

    for batch_idx in range(1, num_batches + 1):
        write_batch(batch_idx, batch_steps, combo, combo_dir, graph_files, system_lmp_path, mass_lines, style, temp)

    submit_script_path = os.path.join(combo_dir, "submit_from.sh")
    with open(submit_script_path, 'w') as f:
        f.write("#!/bin/bash\n")
        f.write("# Usage: ./submit_from.sh <starting_batch_number>\n")
        f.write("set -e\n")
        f.write("COMBO_DIR=\"$(cd \"$(dirname \"$0\")\" && pwd)\"\n")
        f.write(f"NUM_BATCHES={num_batches}\n")
        f.write("START=${1:-1}\n")
        f.write("PREV_JOBID=\"\"\n")
        f.write("for ((i=START; i<=NUM_BATCHES; i++)); do\n")
        f.write("  BATCH_DIR=\"$COMBO_DIR/batch_$i\"\n")
        f.write("  if [ -z \"$PREV_JOBID\" ]; then\n")
        f.write("    JOBID=$(sbatch --parsable --chdir=\"$BATCH_DIR\" \"$BATCH_DIR/lmp.sub\")\n")
        f.write("  else\n")
        f.write("    JOBID=$(sbatch --parsable --dependency=afterok:$PREV_JOBID --chdir=\"$BATCH_DIR\" \"$BATCH_DIR/lmp.sub\")\n")
        f.write("  fi\n")
        f.write("  echo \"Submitted batch_$i as job $JOBID\"\n")
        f.write("  PREV_JOBID=$JOBID\n")
        f.write("done\n")
    os.chmod(submit_script_path, os.stat(submit_script_path).st_mode | stat.S_IEXEC)

    print(f"\n[{combo}] Generated {num_batches} batches ({batch_time_ps} ps each, {total_time_ps} ps total) under {combo_dir}")
    print(f"[{combo}] Submit whole chain: {submit_script_path} 1")
    print(f"[{combo}] Resume after failure at batch N: {submit_script_path} N")


for model in MODELS:
    for sys_name in SYSTEMS:
        generate_for_system(model, sys_name)

Enter the username:  ch24d003
Enter TOTAL run length in ps (e.g. 5000 for 5 ns):  5000
Enter batch chunk length in ps (e.g. 500):  500
Enter the timer timeout PER BATCH in H:MM:SS format (e.g., 23:55:00):  04:00:00
Enter the number of graphs/models considered:  3
Do you want the submission script for CPU or GPU version:  GPU


[DPMD_Interface] Generated graph file path: ./Frozen_Graphs/DPMD_Interface/1/graph1.pb
[DPMD_Interface] Generated graph file path: ./Frozen_Graphs/DPMD_Interface/2/graph2.pb
[DPMD_Interface] Generated graph file path: ./Frozen_Graphs/DPMD_Interface/3/graph3.pb
Successfully generated ./Long_Run/DPMD_Interface/batch_1/in.lammps
Successfully generated ./Long_Run/DPMD_Interface/batch_1/lmp.sub
Successfully generated ./Long_Run/DPMD_Interface/batch_2/in.lammps
Successfully generated ./Long_Run/DPMD_Interface/batch_2/lmp.sub
Successfully generated ./Long_Run/DPMD_Interface/batch_3/in.lammps
Successfully generated ./Long_Run/DPMD_Interface/batch_3/lmp.sub
Successfully generated ./Long_Run/DPMD_Interface/batch_4/in.lammps
Successfully generated ./Long_Run/DPMD_Interface/batch_4/lmp.sub
Successfully generated ./Long_Run/DPMD_Interface/batch_5/in.lammps
Successfully generated ./Long_Run/DPMD_Interface/batch_5/lmp.sub
Successfully generated ./Long_Run/DPMD_Interface/batch_6/in.lammps
Successfully